In [ ]:
from config.config import config, spark
from pyspark.sql.functions import F
from utils.connect import Connection
from utils.padronization import Padronization

from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql import SparkSession

import pandas as pd
import numpy as np
from datetime import datetime 

In [ ]:
def load_treatments(
    glue_context: GlueContext,
    ymd: str,
    database: str,
    table: str,
    padronization: Padronization,
    conn: Connection,
    imports: DynamicFrame
) -> DynamicFrame:

    sql = f"""
        SELECT
            *
        FROM raw
    """
    
    dnf = conn._spark_sql_query(
        sql=sql,
        database=database,
        mapping={"raw":imports}
    )
    
    return dnf

In [ ]:
def load_config_variables():

    glue_context = GlueContext(spark.sparkContext)

    conn = Connection(
        spark=spark,
        glue_context=glue_context
    )

    padronization = Padronization(
        glue_context=glue_context
    )
    
    return glue_context, conn, padronization

In [ ]:
def main():
    
    print("00. Carregando as configurações da rotina")
    glue_context, conn, padronization = load_config_variables()
    ymd = datetime.now().strptime("%Y%m%d")
    database = ''
    input = 'tb_raw_ingestions'
    output = 'tb_trusted_ingestions'
    
    print("01. Carregando obtendo o contexto da tabela do Glue Catalog")
    imports = conn.create_dynamicframe_from_catalog(
        glue_context=glue_context,
        ymd=ymd,
        database=database,
        table=input
    )

    print("02. Executando query")
    treatments = load_treatments(
        glue_context=glue_context,
        ymd=ymd,
        database=database,
        table=input,
        padronization=padronization,
        conn=conn,
        imports=imports
    )

In [ ]:
if __name__ == '__main__':
    main()